# Supplementary Figure S5 — Brain region participation maps at all orders

This notebook generates **Supplementary Figure S5** of the paper:
brain region participation maps for all orders $k = 3, \ldots, 9$,
for both polarities ($\Omega_C > \Omega_{NR}$ and $\Omega_{NR} > \Omega_C$)
in each dataset (MA and DBS).

**Input**: `results/R1_C_region_maps_PRAUC_deltaO.pkl.gz`
(produced by `R1_discrimination/R1_B_region_sampling.ipynb`).

In [ ]:
from pathlib import Path
import os

def ensure_project_root(target_name: str = "high-order-anesthesia") -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == target_name:
        return cwd
    for parent in cwd.parents:
        if parent.name == target_name:
            os.chdir(parent)
            return parent
    raise RuntimeError(f"Could not find '{target_name}' in path.")

ROOT = ensure_project_root()
print(f"Now in: {ROOT.name}")

In [ ]:
import pickle
import gzip
import plotly.io as pio
pio.renderers.default = "notebook_connected"

from src.hoi_anesthesia.plotting import plot_cocomac_region_values

In [ ]:
maps_path = "results/R1_C_region_maps_PRAUC_deltaO.pkl.gz"
with gzip.open(maps_path, "rb") as fh:
    region_maps = pickle.load(fh)
# structure: region_maps[dataset][order][polarity]["region_counts_percent"]
print("Loaded.  Datasets:", list(region_maps.keys()))

## Plot all orders and polarities

In [ ]:
cmap_by_polarity = {"c_gt_nr": "Blues", "nr_gt_c": "Reds"}
variable = "region_counts_percent"

for dataset in ["MA", "DBS"]:
    orders = sorted(region_maps[dataset].keys())
    for order in orders:
        for polarity in ["c_gt_nr", "nr_gt_c"]:
            if polarity not in region_maps[dataset][order]:
                continue
            z_map  = region_maps[dataset][order][polarity][variable]
            pol_label = (r"Ω_C > Ω_NR" if polarity == "c_gt_nr" else r"Ω_NR > Ω_C")
            print(f"\n{dataset} | {pol_label} | order {order}")
            fig = plot_cocomac_region_values(
                z_map,
                cmap_name=cmap_by_polarity[polarity],
                colorbar_title="% of n-plets",
                fontsize=20,
                height=400,
                width=800,
                distance=0.73,
            )
            fig.show()